# Feature Engineering Analysis

**Purpose:** Investigate potential new features that could improve model performance, based on insights from descriptive and inferential analysis.

**Tasks:** 

1. Create new features by combining existing ones (e.g., date-based features like "day of the week").
2. Encode categorical variables: Label encoding, One-Hot encoding, or similar techniques.
3. Feature scaling: Min-Max scaling, Standardization.
4. Feature selection techniques (e.g., based on correlation matrix, mutual information, feature importance).

In [20]:
import json

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm
import statsmodels.stats.api as sms

pd.set_option('display.max_columns', None)

In [21]:
# reading data
df = pd.read_csv(r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\data\raw\fraud test.csv")
df.shape

(555719, 23)

In [22]:
data = df[df.columns[1:]].copy()
data.head(3)

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,21/06/2020 12:14,2.291160e+15,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,19/03/1968,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,21/06/2020 12:14,3.573030e+15,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",17/01/1990,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,21/06/2020 12:14,3.598220e+15,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,Bellmore,NY,11710,40.6729,-73.5365,34496,"Librarian, public",21/10/1970,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0


In [23]:
cols_div_path = r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\notebooks\metadata\Credit_cols_classified.json"
with open(cols_div_path,'r') as file:
    cols_division = json.load(file)
cols_division

{'target_col': 'is_fraud',
 'uniq_cols': ['cc_num', 'trans_num'],
 'num_cols': ['amt', 'city_pop'],
 'cat_cols': ['merchant',
  'category',
  'first',
  'last',
  'gender',
  'street',
  'city',
  'state',
  'job'],
 'loc_cols': ['long', 'lat', 'merch_lat', 'merch_long', 'zip'],
 'time_cols': ['trans_date_trans_time', 'dob', 'unix_time']}

In [24]:
data["amt_per_pop"] = data["amt"] / (data["city_pop"] + 1e-6)
data["log_amt_per_pop"] = np.log1p(data["amt"]) - np.log1p(data["city_pop"])

The feature `log_amt_per_pop` helps capture:

- High transaction amount in small population → suspicious
- Same amount in large city → less suspicious

In [25]:
data.groupby(by = "is_fraud")["log_amt_per_pop"].describe().T

is_fraud,0,1
count,553574.000000,2145.000000
mean,-4.831561,-2.826417
std,2.783637,2.658438
min,-14.184395,-12.233963
25%,-6.640328,-4.592013
50%,-4.432446,-2.374136
75%,-2.776941,-0.800687
max,4.122553,3.559684


`first`, `last` and `street` can be dropped as it will not give any kind of usefull information.

In [26]:
from sklearn.preprocessing import OneHotEncoder,LabelEncoder

In [27]:
cat_cols_encoding = {
    'merchant': "dummie",
    'category': "dummie",
    'gender': "one_hot",
    'city': "label",
    'state': "label",
    'job': "label"
}

In [28]:
# def fit_encode_data(data, col_encod_map, drop_first=True):
#     data = data.copy()
#     encoder_dict = {}

#     for col, encod in col_encod_map.items():
#         if col not in data.columns:
#             continue

#         if encod == "label":
#             le = LabelEncoder()
#             data[col] = le.fit_transform(data[col])

#             encoder_dict[col] = {
#                 "type": "label",
#                 "encoder": le
#             }

#         elif encod == "one_hot":
#             ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
#             transformed = ohe.fit_transform(data[[col]])

#             cols = [f"{col}_{c}" for c in ohe.categories_[0]]
#             if drop_first:
#                 transformed = transformed[:, 1:]
#                 cols = cols[1:]

#             df_ohe = pd.DataFrame(transformed, columns=cols, index=data.index)
#             data = pd.concat([data.drop(columns=[col]), df_ohe], axis=1)

#             encoder_dict[col] = {
#                 "type": "one_hot",
#                 "encoder": ohe,
#                 "columns": cols,
#                 "drop_first": drop_first
#             }

#         elif encod == "dummie":
#             dummies = pd.get_dummies(data[col], prefix=col, drop_first=drop_first,dtype = int)
#             data = pd.concat([data.drop(columns=[col]), dummies], axis=1)

#             encoder_dict[col] = {
#                 "type": "dummie",
#                 "columns": dummies.columns
#             }

#     return data, encoder_dict

In [36]:
def fit_encode_data(data,col_encod_map:dict,drop_first=True):
    encoder_dict = {}
    for col, encod in col_encod_map.items():
        if encod == "dummie":
            dummie_encod = pd.get_dummies
            dummie_encod(data, columns=[col],drop_first=drop_first, dtype=int)
            encoder_dict[col] = dummie_encod
        elif encod == "one_hot":
            one_hot_encod = OneHotEncoder()
            one_hot_encod.fit(data[[col]])
            encoder_dict[col] = one_hot_encod
        elif encod == "label":
            label_encod = LabelEncoder()
            label_encod.fit(data[[col]])
            encoder_dict[col] = label_encod
        else:
            print("Encoder Not Implemented please try from the folowing:\n['dummie','one_hot','label']")
    return encoder_dict


In [37]:
encod_dict = fit_encode_data(data,cat_cols_encoding)

MemoryError: Unable to allocate 2.87 GiB for an array with shape (555719, 693) and data type int64

In [38]:
data.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,amt_per_pop,log_amt_per_pop
0,21/06/2020 12:14,2.291160e+15,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,19/03/1968,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0,0.000009,-11.366725
1,21/06/2020 12:14,3.573030e+15,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",17/01/1990,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0,0.098808,-2.284920
2,21/06/2020 12:14,3.598220e+15,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,Bellmore,NY,11710,40.6729,-73.5365,34496,"Librarian, public",21/10/1970,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0,0.001197,-6.704313
3,21/06/2020 12:15,3.591920e+15,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,Titusville,FL,32780,28.5697,-80.8191,54767,Set designer,25/07/1987,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0,0.001096,-6.799168
4,21/06/2020 12:15,3.526830e+15,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,Falmouth,MI,49632,44.2529,-85.0170,1126,Furniture designer,06/07/1955,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0,0.002833,-5.594614


In [40]:
encod_dict

{'merchant': <function pandas.core.reshape.encoding.get_dummies(data, prefix=None, prefix_sep: 'str | Iterable[str] | dict[str, str]' = '_', dummy_na: 'bool' = False, columns=None, sparse: 'bool' = False, drop_first: 'bool' = False, dtype: 'NpDtype | None' = None) -> 'DataFrame'>,
 'category': <function pandas.core.reshape.encoding.get_dummies(data, prefix=None, prefix_sep: 'str | Iterable[str] | dict[str, str]' = '_', dummy_na: 'bool' = False, columns=None, sparse: 'bool' = False, drop_first: 'bool' = False, dtype: 'NpDtype | None' = None) -> 'DataFrame'>,
 'gender': OneHotEncoder(),
 'city': LabelEncoder(),
 'state': LabelEncoder(),
 'job': LabelEncoder()}

In [51]:
for col, encoder in encod_dict.items():
    if callable(encoder):  # handles get_dummies
        print("In dummy encoder")
        dummies = encoder(data[col], prefix=col)
        data = pd.concat([data.drop(columns=[col]), dummies], axis=1)

    else:
        print("Not in dummy encoder")
        
        if hasattr(encoder, "transform"):
            transformed = encoder.transform(data[[col]])
            
            # Handle OneHotEncoder separately (returns multiple columns)
            if transformed.ndim > 1 and transformed.shape[1] > 1:
                new_cols = [f"{col}_{i}" for i in range(transformed.shape[1])]
                transformed_df = pd.DataFrame(transformed, columns=new_cols, index=data.index)
                data = pd.concat([data.drop(columns=[col]), transformed_df], axis=1)
            else:
                data[col] = transformed

In dummy encoder
In dummy encoder


MemoryError: Unable to allocate 33.9 MiB for an array with shape (8, 555719) and data type float64

In [34]:
def encode_data(data, encoder_dict):
    data = data.copy()

    for col, info in encoder_dict.items():
        if info["type"] == "label":
            le = info["encoder"]
            data[col] = le.transform(data[col])

        elif info["type"] == "one_hot":
            ohe = info["encoder"]
            transformed = ohe.transform(data[[col]])

            cols = [f"{col}_{c}" for c in ohe.categories_[0]]

            if info["drop_first"]:
                transformed = transformed[:, 1:]
                cols = cols[1:]

            df_ohe = pd.DataFrame(transformed, columns=cols, index=data.index)
            data = pd.concat([data.drop(columns=[col]), df_ohe], axis=1)

        elif info["type"] == "dummie":
            dummies = pd.get_dummies(data[col], prefix=col)

            # Align with training columns
            for c in info["columns"]:
                if c not in dummies:
                    dummies[c] = 0

            dummies = dummies[info["columns"]]

            data = pd.concat([data.drop(columns=[col]), dummies], axis=1)

    return data

In [35]:
data = encode_data(data,encod_dict)
data.head()

TypeError: 'function' object is not subscriptable